# Reaching what you can't wait for — steered MD, umbrella sampling & free energies
Notebooks 1–2 ran *unbiased* Trp-cage MD, and §2.6 showed the hard truth: on the timescale a notebook — or even our 10 ns reference — can afford, the interesting events are barely sampled, and the defining cage never opens at all. When you can't wait for a transition, you **bias** the simulation to make it happen — then, because you added the bias and know it exactly, **divide it back out** of the statistics to recover the equilibrium free energy *your model* implies along that coordinate. (How completely it can be divided out — never a given — is §3.5's running theme.) That family of tricks is **enhanced sampling**.

There are several families (replica exchange / tempering, metadynamics, adaptive biasing force, …); this notebook demonstrates the most direct one — **biasing a collective variable (CV)**: pick a coordinate that captures the transition and put a harmonic restraint on it. We use that restraint in two ways:
- **Steered MD** (§3.3–3.4): *ramp* the restraint's center outward to drive the cage open — fast, but it distorts the thermodynamics.
- **Umbrella sampling + MBAR** (§3.5): *hold* the restraint at a ladder of fixed centers spanning closed→open, then statistically stitch the windows back together — removing the bias — into a **free-energy profile** along the coordinate, called a **potential of mean force (PMF)**: the effective free energy as a function of the CV, with every *other* degree of freedom averaged out.

**Roadmap:** unbiased MD hits a wall (§3.0–3.1) → a steered pull drives the cage open, judged by *independent* referees (§3.2–3.4) → umbrella windows + MBAR turn the push into a PMF (§3.5) → and a second coordinate, the Asp9–Arg16 salt bridge, that plain MD *does* sample — giving us ground truth to validate the whole method against (§3.6). Throughout, the event is the Trp-cage's namesake: prying the cage open around Trp6 — peeling the poly-proline lid back rather than ejecting the indole (Trp6 barely moves; see §3.4).

In [ ]:
#@title Environment on-ramp (imports + modules)
# --- environment on-ramp: make sure the MD stack + the modules are importable in THIS kernel ---
# (identical pattern to 01/02 -- check core deps, provision on Colab, else point at the 'MD tutorial' env)
import importlib.util, sys, os, subprocess, glob
_missing = [m for m in ("openmm", "pdbfixer", "mdtraj", "py3Dmol", "pymbar") if importlib.util.find_spec(m) is None]
if _missing and "google.colab" in sys.modules:                          # Colab: provision the stack
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openmm", "pdbfixer", "mdtraj", "py3Dmol", "pymbar"], check=False)
    _missing = [m for m in _missing if importlib.util.find_spec(m) is None]
if _missing:                                                            # still missing -> almost always the WRONG KERNEL
    raise SystemExit(f"Missing in this kernel: {_missing}. Select your MD-tutorial conda kernel "
                     "(Kernel > Change Kernel). If it isn't built yet: conda env create -f environment.yml; "
                     "if it exists but is stale: conda env update -f environment.yml.")
_BASE = os.environ.get("MDTUTORIAL_BASE", "https://raw.githubusercontent.com/todd471/MD_tutorial/main")
for _mod in ("mdtutorial.py", "mdtviz.py", "pull_screen.py"):           # grab the shipped modules if absent
    if not os.path.exists(_mod) and importlib.util.find_spec(_mod[:-3]) is None:
        import urllib.request
        try: urllib.request.urlretrieve(f"{_BASE}/{_mod}", _mod); print("fetched", _mod)
        except Exception as e: print("Place", _mod, "next to this notebook.", e)
import numpy as np, mdtraj as md, matplotlib.pyplot as plt
import openmm, openmm.app as app
import mdtutorial as mdt, mdtviz, pull_screen as steer     # steered-MD/umbrella/MBAR harness + viewers
REF = "reference_10ns"
def ensure_file(relpath):                        # fetch ONE repo file into place if absent (Colab has no local reference)
    if not os.path.exists(relpath):
        os.makedirs(os.path.dirname(relpath) or ".", exist_ok=True)
        import urllib.request; urllib.request.urlretrieve(f"{_BASE}/{relpath}", relpath)
    return relpath
REF_SEEDS = [2021, 2022, 2023, 2024, 2025, 2026]   # reference seeds; ref_trajs() pulls only the ones a cell actually uses
def ref_trajs(seeds=None):                       # sorted reference-trajectory paths, fetched ON DEMAND (only what's needed)
    return [ensure_file(os.path.join(REF, "md_output", f"traj_{s}.dcd")) for s in (seeds or REF_SEEDS)]
TOP = ensure_file(os.path.join(REF, "structures", "protein.pdb"))

In [ ]:
# --- Knobs: the levers you actually turn. Each is explained; defaults give a fast-but-honest first run. ---
SOLVENT  = "explicit"     # "explicit" = TIP3P water + PME (the physically honest model; used by every run below).
                          # "implicit" = GB continuum, no water: ~10-20x faster -> switch to it when you want to
                          #              pull FARTHER or run LONGER (past ~200 ps) than explicit affords here.
TEMP      = 300           # single simulation temperature, K (300 K ≈ Zhou's reference). We saw NO temperature trend 300–330 K, so §3.3 shows independent REPEATS at one T rather than a temperature sweep.
N_REPEATS = 3             # independent steered-pull repeats in §3.3 — fresh random velocities each -> run-to-run spread, not a temperature axis.  [dev: drop to 2 to go faster]
PULL_PS  = 200            # length of each steered pull, ps (§3.3 CV screen and §3.4 movie). Longer = smoother, slower.  [dev value; ship default ≈100]
NWIN     = 12             # umbrella windows (§3.5); over the shorter UMB_SPAN below they sit ~0.3 Å apart -> good overlap.
UMB_SPAN = 0.32           # umbrella ladder width, nm (~folded+3.2 Å ≈ lid-off). Deliberately SHORTER than the steered-pull ramp
#                           so the windows stay on the actual cage-opening, not the 'drag the tail through water' stretch past ~8 Å.
WIN_PS   = 40             # sampling ps per umbrella window (§3.5); more frames + better equilibration -> less inflated PMF.
EQUIL_PS = 12             # equilibration ps before each window records (§3.5); let it settle at its new center first.
_avail = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
PLATFORM = next((p for p in ("CUDA", "OpenCL", "CPU") if p in _avail), "CPU")   # CUDA=Colab GPU, OpenCL=Apple GPU, CPU=fallback
print(f"platform {PLATFORM} | {SOLVENT} solvent | T {TEMP} K | {N_REPEATS} pull repeats | pull {PULL_PS} ps | umbrella {NWIN}x{WIN_PS} ps")

## 3.0  Motivation: a force on a single mode
Enhanced sampling means **choosing one coordinate and pushing on it.** Here is the coordinate for the cage — the indole→poly-Pro distance — with its free energy **measured from the unbiased reference** (all six seeds, −k_BT·ln P), before we run anything.

**Where does a free energy even come from, in an MD run?** Only from *how much time the system spends* at each value of the coordinate. Histogram the trajectory along the CV and you get **P(CV)** — the equilibrium probability of finding the system there — and the free energy is simply **G(CV) = −k_BT·ln P(CV)** (the *Boltzmann inversion*). Heavily-visited spots (high P) are low free energy; rarely-visited spots (low P) are high; a PMF is nothing more than −k_BT·ln of a population. The catch, and the reason this whole section exists: a **barrier is high free energy → low P → almost never visited**, so unbiased MD barely samples it — which is exactly why the curve below falls apart past the well. Enhanced sampling *forces* the system to visit those rare spots, and **MBAR's job is to reconstruct the unbiased P** from the biased samples.

A deep **folded well** (~4.7 Å) rises steadily — but read the **seed-to-seed band** (shaded): the six seeds agree tightly near the well and then **fan apart past ~6 Å**, because out there only *one* of the six seeds ever really strays — five never leave the folded well, and even the lone straying one never reaches a fully-open cage — so the curve past ~6 Å is a **single trajectory's excursion (n = 1)**, not a converged free energy (read the error band, not just the mean — an apparent shoulder in a region only one seed samples is sampling noise, not a metastable state). Past ~7.7 Å the unbiased trajectories **run out** entirely — they never reach the fully-open cage, so the free energy beyond is *unknown* (the shaded *here-be-dragons* region). A **moving harmonic restraint** ½k(x−r₀)² is the handle we bolt onto this one coordinate — ramping r₀ outward drags the system past the wall (§3.3), and removing the bias afterward is how we'll *try* to recover the free energy out there (§3.5).

In [ ]:
#@title §3.0 — reference free-energy figure (code)
# Ground the motivation in DATA: free energy along the real cage-opening coordinate (indole->poly-Pro),
# measured from the unbiased reference (all seeds, -kT ln P). No simulation yet -- just what we already know.
_ring = md.load(TOP).topology.select('resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2')
_lid  = md.load(TOP).topology.select('resSeq 17 18 19 and name CA')
_cvf  = lambda tr: np.linalg.norm(tr.xyz[:, _ring].mean(1) - tr.xyz[:, _lid].mean(1), axis=1) * 10
_per  = [_cvf(md.load(f, top=TOP)) for f in ref_trajs()]                        # per seed (Å), for the seed-to-seed band
_kT   = 0.00831446261815324 * TEMP
_mc, _G, _band, _ir = steer.reference_pmf(_per, TEMP, nbins=30)                 # honest -kT lnP + flaring band + contiguous mask
#                                                                              (single source: the SAME helper backs the 3.5c cross-check)
_XR = 10.6                                                                     # right edge of plot / grey box (room for the dragons)
_wall = _mc[_ir]; _wy = _G[_ir] / _kT; _box = _XR - _wall                       # wall x, wall y (blue endpoint), box width
plt.figure(figsize=(7.6, 4.3))
plt.plot(_mc, _G / _kT, 'o-', color='navy', ms=4, label='free energy, measured from unbiased MD')
plt.fill_between(_mc, (_G - _band) / _kT, (_G + _band) / _kT, color='navy', alpha=0.15,
                 label='reference uncertainty (seed spread + sampling; flares where seeds thin out)')
plt.axvspan(_wall, _XR, color='0.93')
# extrapolate HONESTLY from the trend of the last few measured points (departs tangent to the blue -- no kink),
# and show TWO divergent scenarios so the reader sees we genuinely don't know what's past the wall.
_nfit = 6; _lo0 = max(0, _ir - _nfit + 1)
_xf = _mc[_lo0:_ir + 1]; _yf = (_G / _kT)[_lo0:_ir + 1]
_m = float(np.polyfit(_xf, _yf, 1)[0])                                          # measured slope at the wall (k_BT/Å)
_xw = np.linspace(_wall, _XR, 60); _tt = _xw - _wall
_gA = np.clip(_wy + _m * _tt, None, 10.3)                                       # A: the barrier keeps climbing
_climb = _wy + _m * _tt; _frac = _tt / _box                                     # tangent departure (matches A near the wall)
_basin = 2.6 + 4.4 * np.clip((_frac - 0.5) / 0.5, 0, 1) ** 2                    # basin floor, rising to a far wall (the upswing)
_wgt = 1.0 / (1.0 + np.exp(-(_tt - 0.24 * _box) / (0.07 * _box)))               # blend: climb near the wall -> basin far out
_gB = np.clip((1 - _wgt) * _climb + _wgt * _basin, None, 10.3)                  # B: departs tangent, crests a small barrier, dips into a basin with a far wall
plt.plot(_xw, _gA, ':', color='0.4', lw=2.0, label='extrapolation A: barrier keeps climbing')
plt.plot(_xw, _gB, color='#b5651d', lw=1.8, ls=(0, (5, 2)), label='extrapolation B: a hidden open basin')
for _fx, _qy, _qs, _qr in [(0.28, 9.3, 14, -8), (0.40, 8.6, 11, 10), (0.52, 9.5, 13, 0), (0.62, 8.8, 12, -11),
                           (0.70, 7.2, 12, 8), (0.82, 7.5, 11, -6), (0.58, 6.5, 10, 12)]:   # scattered clear of legend, between the guesses
    plt.text(_wall + _fx * _box, _qy, '?', fontsize=_qs, color='0.5', ha='center', va='center', rotation=_qr)
plt.text(_wall + 0.40 * _box, 0.12, 'past the wall the free energy is *unmeasured* —\ntwo honest guesses, not data (§3.5 tries to reach it)',
         ha='center', va='bottom', fontsize=7.5, color='0.4')
_sx = _wall + np.linspace(0.10, 0.44, 150) * _box; _sy = 1.7 + 0.24 * np.sin((_sx - _sx[0]) * 7.2)   # here-be-dragons sea serpent
plt.plot(_sx, _sy, color='0.5', lw=1.7, solid_capstyle='round')
for _fx in np.linspace(_sx[8], _sx[-14], 8):                                    # dorsal fins along the back
    _fy = 1.7 + 0.24 * np.sin((_fx - _sx[0]) * 7.2); _fw = 0.024 * _box
    plt.fill([_fx - _fw, _fx, _fx + _fw], [_fy, _fy + 0.5, _fy], color='0.5', lw=0)
plt.plot(_sx[-1], _sy[-1], 'o', color='0.5', ms=7)                              # head
plt.plot([_sx[-1] + 0.03], [_sy[-1] + 0.08], '.', color='white', ms=3.5)       # eye
plt.plot([_sx[-1] + 0.03, _sx[-1] + 0.2], [_sy[-1] - 0.05, _sy[-1] - 0.14], color='firebrick', lw=1.0)  # tongue
plt.annotate('to see past the wall, we push the system\ntoward it — out of the well  (§3.3)',
             xy=(_wall, _wy), xytext=(4.35, 6.7), fontsize=8.5, color='firebrick',
             arrowprops=dict(arrowstyle='->', color='firebrick', lw=1.4, connectionstyle='arc3,rad=-0.2'))
plt.xlim(4, _XR); plt.ylim(-0.6, 10.5); plt.xlabel('indole→poly-Pro distance (Å)'); plt.ylabel('free energy (k$_B$T)')
plt.title('The one coordinate we push on — grounded in the real data'); plt.legend(fontsize=8, loc='upper left'); plt.show()
print(f'From the unbiased reference: folded well at {_mc[np.nanargmin(_G)]:.2f} Å; the seeds agree only near '
      f'the well and fan apart past ~6 Å (band), where sampling thins to a single seed -- never the open cage.')

## 3.1  The timescale wall — and Zhou's clock
Unbiased MD **did not open the cage**: across the 6-seed 10 ns reference, Trp6's side-chain SASA never exceeds ~67 Å² (a fully exposed Trp is ~200–250) and Cα-RMSD stays under 3.3 Å — the molecule sits deep in the folded well the whole time. Yet the event isn't fictional, it just won't wait for us: **Zhou (2003)** reports a metastable intermediate on Trp-cage's folding route whose Asp9–Arg16 salt bridge breaks and re-forms in 20-ns simulations — right on top of our own 6 × 10 ns window. The transition lives at this timescale; we just don't catch it spontaneously. So we **force** it — starting folded and prying the cage open, the hypothetical reverse of that last folding step. **Can we run Zhou's folding backwards by pushing?**

In [ ]:
#@title §3.1 — the timescale wall (figure code)
# See the wall, don't just read it. Over a full 10 ns unbiased seed -- ~50x longer than a steered pull --
# how far does Trp6 actually move? Overlay its first and last frame, and quote the displacement + exposure.
_seed = md.load(ref_trajs()[0], top=TOP)
_seed.superpose(_seed, 0)                                          # align out overall tumbling
_sc = _seed.topology.select('resSeq 6 and sidechain'); _trp = _seed.topology.select('resSeq 6')
_sasa = md.shrake_rupley(_seed[::20], mode='atom')[:, _sc].sum(1) * 100
_move = md.rmsd(_seed[-1], _seed[0], atom_indices=_trp)[0] * 10    # Trp6 heavy-atom RMSD, first -> last (Å)
os.makedirs('es_structures', exist_ok=True)
_seed[0].save_pdb('es_structures/ref_first.pdb'); _seed[-1].save_pdb('es_structures/ref_last.pdb')
print(f'Trp6 SASA over the 10 ns seed: max {_sasa.max():.0f} Å²  (this seed; ≤~67 Å² across all six -- fully exposed Trp ~200-250 Å²) -- the cage stays shut.')
print(f'First -> last frame, Trp6 moved just {_move:.1f} Å (heavy-atom RMSD). Over ~50x the length of a steered pull, it only breathes.')
print('   BLUE = first frame,  ORANGE = last frame -- the two Trp6 sticks nearly coincide.')
mdtviz.overlay_view('es_structures/ref_first.pdb', 'es_structures/ref_last.pdb', highlight_resi=6)

## 3.2  Where to push, and how to measure it
**A tutorial choice, stated plainly.** We are *not* deriving the true reaction coordinate for cage opening — that's a hard, separate problem of its own, needing dedicated machinery well beyond a tutorial. We screened a handful of simple, restrainable distances against Trp6 exposure (SASA) on the unbiased reference (`cage_cv_screen.py`, plotted below), and two lessons fall out:

- **Correlation isn't enough.** The single best-correlating distance is indole→*cage-centroid* (r≈0.55) — but that centroid is an average over the pocket-lining residues, which spread and reorganize *as the cage opens*. The moment you start prying the cage apart, the reference you're measuring against stops meaning much: a drifting, ill-conditioned coordinate.
- **A CV must stay well-defined across the whole transition.** So we bias **indole→poly-Pro lid** (r≈0.47, a close second): the distance from the **Trp6 indole ring** to the Cα *centroid* of the **Pro17–Pro18–Pro19** segment. The proline that actually caps the indole is **Pro18**, at the center of that triplet (indole→Pro18 alone tracks exposure at r≈0.27); the centroid sits essentially on it (≈4.8 Å folded, the same as the direct indole→Pro18 distance). Flanking Pro18 with Pro17 and Pro19 doesn't move the landmark — it makes it a *rigid three-residue anchor* that stays put whether the cage is shut or open, and it correlates with opening **better than the single Pro18 contact** (0.47 vs 0.27). §3.3 confirms steering it opens the cage *locally* (Trp6 exposes; global RMSD stays low).

It's a reasonable, teachable choice — picked for being well-defined and locally-actionable, not for being *the* coordinate.

**How the force is applied — steered MD:** a harmonic restraint U = ½k(CV − r₀)² whose center r₀ is ramped outward, dragging the CV and peeling the poly-proline lid off the indole. **A subtlety worth pausing on:** the CV isn't a bond between two atoms — it's the distance between two *centroids* (the mass-weighted centers of the indole-ring atoms and of the three proline Cα atoms), and a centroid is a computed point, not a physical object. So what does the force actually push on? The restraint's gradient acts on *every atom in both groups*, distributed by mass, so the net effect slides the whole indole ring away from the whole proline cluster along the line joining their centers — no single atom is the anchor. That force spread over many atoms is exactly what makes it a *collective* variable (and why the group choice is forgiving: you bias an aggregate coordinate, not one fragile contact). The dashed lines in the schematic map and 3-D view below mark precisely these centroid-to-centroid distances. **Independent referees (never biased):** Trp6 **SASA** (cage exposure), the **Asp9–Arg16 salt-bridge** distance, and **Cα-RMSD**. Their job isn't to score a 'success' — it's to watch what pushing this *one* coordinate does to the rest of the molecule, measured by things we didn't push on (never read off only what you biased).

In [ ]:
# WHY indole->polyPro? The DATA: how well does each candidate distance track Trp6 exposure (SASA) across
# the 6-seed unbiased reference (cage_cv_screen.py)? Correlation ranks them; well-definedness picks the winner.
_RING = 'resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2'
_cen = lambda t, s: t.xyz[:, t.topology.select(s), :].mean(1)
_S, _C = [], {k: [] for k in ['indole→polyPro lid', 'salt bridge (Asp9-Arg16)', 'indole→cage centroid',
                              'indole→Tyr3 ring', 'Trp6 burial depth']}
for _f in ref_trajs():
    _t = md.load(_f, top=TOP)[::10]
    _S.append(md.shrake_rupley(_t, mode='atom')[:, _t.topology.select('resSeq 6 and sidechain')].sum(1) * 100)
    _r = _cen(_t, _RING)
    _C['indole→polyPro lid'].append(np.linalg.norm(_r - _cen(_t, 'resSeq 17 18 19 and name CA CB CG'), axis=1) * 10)
    _asp = _t.topology.select('resSeq 9 and name OD1 OD2'); _arg = _t.topology.select('resSeq 16 and name NH1 NH2 NE')
    _C['salt bridge (Asp9-Arg16)'].append(md.compute_distances(_t, [(a, b) for a in _asp for b in _arg]).min(1) * 10)
    _C['indole→cage centroid'].append(np.linalg.norm(_r - _cen(_t, 'resSeq 3 11 12 14 18 19 and name CA'), axis=1) * 10)
    _C['indole→Tyr3 ring'].append(np.linalg.norm(_r - _cen(_t, 'resSeq 3 and name CG CD1 CD2 CE1 CE2 CZ'), axis=1) * 10)
    _C['Trp6 burial depth'].append(np.linalg.norm(_cen(_t, 'resSeq 6 and sidechain') - _cen(_t, 'name CA'), axis=1) * 10)
_S = np.concatenate(_S); _C = {k: np.concatenate(v) for k, v in _C.items()}
_ranked = sorted(_C, key=lambda k: -abs(np.corrcoef(_C[k], _S)[0, 1])); _pick = 'indole→polyPro lid'
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8), layout='constrained')
_rs = [abs(np.corrcoef(_C[k], _S)[0, 1]) for k in _ranked]
ax[0].barh(range(len(_ranked)), _rs, color=['firebrick' if k == _pick else '0.7' if k == _ranked[0] else 'steelblue' for k in _ranked])
ax[0].set_yticks(range(len(_ranked))); ax[0].set_yticklabels(_ranked, fontsize=8); ax[0].invert_yaxis()
ax[0].set_xlabel('|correlation| with Trp6 SASA'); ax[0].set_title('correlation ranking', fontsize=9); ax[0].set_xlim(0, 0.85)
_yi = {k: i for i, k in enumerate(_ranked)}
ax[0].text(_rs[_yi[_pick]] + 0.01, _yi[_pick], ' ← we bias this\n    (well-defined, local)', va='center', fontsize=7, color='firebrick')
ax[0].text(_rs[0] + 0.01, 0, ' top r, but a\n    drifting reference', va='center', fontsize=7, color='0.4')
for _a, _k in zip(ax[1:], ['indole→polyPro lid', 'salt bridge (Asp9-Arg16)']):
    _a.hist2d(_C[_k], _S, bins=50, cmap='turbo', cmin=1)
    _a.set_xlabel('%s  (Å)' % _k); _a.set_ylabel('Trp6 SASA (Å²)')
    _a.set_title('%s\nr = %+.2f' % (_k.split(' (')[0], np.corrcoef(_C[_k], _S)[0, 1]), fontsize=9)
plt.show()
print('ranked by |r| with SASA: ' + ' | '.join('%s %+.2f' % (k, np.corrcoef(_C[k], _S)[0, 1]) for k in _ranked))
print(f'-> we bias {_pick} (r={np.corrcoef(_C[_pick], _S)[0,1]:+.2f}): not the top correlate, but well-defined, and it opens the cage locally (3.3).')

In [ ]:
#@title §3.2 — locator diagram (code)
# A labelled schematic MAP of the same four groups on a 2-D projection of the fold: the biased CV and the
# Asp9–Arg16 salt bridge as dashed lines, each group tethered to its backbone Cα, and a pictogram for every
# observable (biased CV, Trp6 SASA, salt bridge, global Cα-RMSD). Read it alongside the 3-D render below.
mdtviz.cv_locator_map(TOP)

In [ ]:
#@title §3.2 — 3-D CV view + setup (code)
pdb = app.PDBFile(TOP); mdtop = md.load(TOP).topology; x0 = md.load(TOP)
sel = steer.selections(mdtop, x0)                 # referee atom selections
CVS = steer.cage_cvs(mdtop, pdb.positions)        # the 5 candidate biasing CVs
ring = mdtop.select('resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2'); lid = mdtop.select('resSeq 17 18 19 and name CA')
cv_polyPro = lambda tr: np.linalg.norm(tr.xyz[:, ring].mean(1) - tr.xyz[:, lid].mean(1), axis=1) * 10
print('candidate CVs:', list(CVS)); print(f'folded indole→polyPro distance = {cv_polyPro(x0)[0]:.1f} Å')
print('   ORANGE = Trp6 indole  ·  BLUE = poly-Pro lid (Pro17-18-19 Cα; its centroid sits on Pro18, the stacker)  ·  RED/GREEN = Asp9/Arg16 salt bridge')
mdtviz.cv_groups_view(TOP, [('6', 'orangeCarbon'), ('17-19', 'blueCarbon'), ('9', 'redCarbon'), ('16', 'greenCarbon')],
    cv_pairs=[('resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2', 'resSeq 17 18 19 and name CA', '0x000000'),  # indole↔polyPro CV (biased)
              ('resSeq 9 and name OD1 OD2', 'resSeq 16 and name NH1 NH2 NE', '0x000000')])                        # Asp9↔Arg16 salt bridge

## 3.3  Results — the cage opens on one coordinate, not the other
The panels below make the case directly. Each **row** biases one coordinate (left) and watches the **independent** Trp6-SASA referee (right), on a **matched scale** so the 67 Å² unbiased ceiling lines up across both:

- **Bias indole→poly-Pro lid** and SASA climbs with the pull, past the ceiling — the cage opens — while global Cα-RMSD stays low (a *local* opening, not unfolding; see the printout).
- **Bias the salt bridge** and its Asp9–Arg16 distance opens right up, but Trp6 SASA barely moves: the cage stays shut. Breaking the salt bridge is **not sufficient** to open the cage.

So the salt bridge is **coupled** to cage opening (part of the folded latch) but is **not the driver** — the reaction-coordinate lesson: a coordinate can correlate with a transition without being what *causes* it. These are short, fast pulls — a few **independent repeats** at a single temperature — so read them qualitatively; the converged, quantitative statement is the free-energy profile in §3.5.

> **Heads-up — this is the slowest cell in the tutorial.** It runs `N_REPEATS` steered pulls of `PULL_PS` ps for *each* of the two coordinates (both knobs are set at the top), i.e. several hundred picoseconds of explicit-solvent MD several times over — expect a few minutes on a laptop or Colab GPU (much longer CPU-only). It **prints a line as each pull finishes**, so a quiet cell is still working, not frozen.

In [ ]:
# Steer each CV and watch TWO things over time: the coordinate we push (left) and the INDEPENDENT
# referee, Trp6 SASA (right). Push indole->polyPro and SASA climbs with it; push the salt bridge and its
# distance opens but SASA barely moves -- coupled to cage opening, not the driver. Matched SASA axes.
fig, ax = plt.subplots(2, 2, figsize=(10, 6.6), sharex=True, layout='constrained')
_lss = ['-', '--', ':', '-.']; _sasa_all = []
for _i, (_cv, _note) in enumerate([('indole→polyPro lid', 'opens the cage'), ('salt bridge', 'does not open the cage')]):
    _fac, _delta = CVS[_cv]
    for _j in range(N_REPEATS):                                                     # independent repeats at ONE T (fresh velocities each)
        _sa, _sb, _rmsd, _traj = steer.run_pull(pdb, mdtop, _fac, _delta, TEMP, PULL_PS, PLATFORM, sel, solvent=SOLVENT)
        _t = np.linspace(0, PULL_PS, len(_sa))
        _biased = cv_polyPro(_traj) if _cv == 'indole→polyPro lid' else _sb        # the coordinate we actually pushed
        _ls = _lss[_j % 4]
        ax[_i, 0].plot(_t, _biased, _ls, label=f'repeat {_j + 1}'); ax[_i, 1].plot(_t, _sa, _ls, label=f'repeat {_j + 1}')
        _sasa_all.append(_sa)
        print(f'{_cv:20s} repeat {_j + 1}/{N_REPEATS}: pushed {_biased[0]:.1f}->{_biased.max():.1f} Å | Trp6 SASA {_sa[0]:.0f}->{_sa.max():.0f} Å² | RMSD ->{_rmsd[-1]:.1f} Å   [pull done]')
    ax[_i, 0].set_ylabel('%s\n(Å, the coordinate pushed)' % _cv, fontsize=8); ax[_i, 1].set_ylabel('Trp6 SASA (Å²)')
    ax[_i, 1].axhline(67, color='0.6', ls=':', lw=1)                                # unbiased 10 ns exposure ceiling
    ax[_i, 0].text(0.03, 0.93, 'bias %s → %s' % (_cv, _note), transform=ax[_i, 0].transAxes, fontsize=9, va='top',
                   color=('firebrick' if _i == 0 else 'steelblue'))
_ymax = max(s.max() for s in _sasa_all) * 1.05
for _a in ax[:, 1]:
    _a.set_ylim(0, _ymax)                                                           # matched SASA axis -> honest comparison
ax[0, 0].set_title('the coordinate we push', fontsize=9); ax[0, 1].set_title('Trp6 SASA — independent referee', fontsize=9)
ax[1, 0].set_xlabel('time (ps)'); ax[1, 1].set_xlabel('time (ps)'); ax[0, 1].legend(fontsize=7, title='independent repeats')
plt.show()

## 3.4  What it looks like on the molecule
A **controlled comparison**, not a loaded one: both trajectories are the **same length**, from the **same folded start**, in the **same explicit solvent** — the *only* difference is that the right one has the steering force applied. **Left (unbiased):** the cage breathes but stays folded. (The quantitative claim that unbiased MD *never* opens it is §3.1's 10 ns reference — this short matched clip just shows the folded motion under identical conditions, so the two panels differ by the force alone, not by how long we watched.) **Right (steered `indole→polyPro`):** as the restraint center ramps outward the force works the **poly-Pro lid loose** — the lid peeling back is what you see move, while Trp6 itself **barely shifts**. That local loosening, completing late in the pull as the force finally wins, is what we've been calling 'cage opening.' Trp6 is orange sticks; the faint grey ghost is its folded starting position.

In [ ]:
#@title §3.4 — before/after movie (code)
# Two aligned trajectories (subsampled): unbiased (stays shut) vs steered (opens), animated side by side.
_shut = steer.run_unbiased(pdb, mdtop, TEMP, PULL_PS, PLATFORM, save_ps=2.0, solvent=SOLVENT)  # same length as the pull -> equal-duration side-by-side
_fac, _delta = CVS['indole→polyPro lid']
*_, _open = steer.run_pull(pdb, mdtop, _fac, _delta, TEMP, PULL_PS, PLATFORM, sel, save_ps=2.0, solvent=SOLVENT)
steer.save_aligned(_shut, x0, 'es_structures/traj_shut.pdb')      # superpose -> shared folded orientation
steer.save_aligned(_open, x0, 'es_structures/traj_open.pdb')
print('   LEFT: unbiased — cage stays shut            RIGHT: steered — cage opens (the lid peels back; Trp6 barely shifts)')
mdtviz.dual_view('es_structures/traj_shut.pdb', 'es_structures/traj_open.pdb', highlight_resi=6,
                 cv_pairs=[('resSeq 6 and name CG CD1 CD2 NE1 CE2 CE3 CZ2 CZ3 CH2',
                           'resSeq 17 18 19 and name CA', '0x111111')])   # animated indole→polyPro CV on both panels

## 3.5  From a push to a free energy — umbrella sampling + MBAR
This is the most abstract section, so we build it in stages — nobody absorbs MBAR on the first read: **explain → sample → look at the raw windows → explain MBAR → stitch two windows by hand → do all of them → read the result.**

**Why the pull's free energy is wrong.** A fast steered pull is a *non-equilibrium* process: we yank the restraint faster than the surroundings can relax, so some of the work we did went into friction and dissipation, not into the free energy. That dissipated work is never negative *on average*, so the **average pull work always *over*estimates the true free-energy difference** — the more irreversible the pull, the worse the overshoot. (This is the non-equilibrium work relation of **Jarzynski**, *Phys. Rev. Lett.* **78**, 2690 (1997), doi:10.1103/PhysRevLett.78.2690: a specially-weighted exponential average of *many* pulls does recover ΔF exactly, but it converges painfully slowly.) So a pull can *drive* the transition, but we can't read ΔF off it.

**Umbrella sampling — tile the path instead of racing across it.** Rather than *ramping* one restraint continuously, we hold it at a **ladder of fixed centers** r₀ spanning closed→open (`centers`) — the umbrella-sampling idea of **Torrie & Valleau**, *J. Comput. Phys.* **23**, 187 (1977), doi:10.1016/0021-9991(77)90121-8. Each center pins the system in a thin slice of the coordinate it would otherwise never visit; sample every slice and together they **tile the whole path, barrier included**. Concretely, we set each window's restraint at its center, let it **settle there** for `equil_ps`, then record the CV for `sample_ps` (`run_umbrella`). Each window's samples are *biased* — tugged toward its center — not the equilibrium distribution on their own, which is exactly what MBAR undoes. It's cheap: a short settle and a short sample per window — how cheap, and what that costs us, is the honest subject of §3.5d. **The sampling cell below takes a bit** — it's real MD, one window after another.

In [ ]:
# SAMPLE: run the umbrella ladder. Hold the restraint at each of NWIN centers; each window settles
# then samples the CV -> a stack of BIASED distributions (windows). No statistics yet.
fac, _ = CVS['indole→polyPro lid']; K = 4000.0; T = TEMP           # (the steered-pull delta is NOT reused -> see UMB_SPAN)
folded = float(np.linalg.norm(x0.xyz[0, ring].mean(0) - x0.xyz[0, lid].mean(0)))   # nm
centers = np.linspace(folded, folded + UMB_SPAN, NWIN)             # ladder stops near lid-off, not out in the stretch regime
windows = steer.run_umbrella(pdb, fac, centers, T, PLATFORM, k=K, equil_ps=EQUIL_PS, sample_ps=WIN_PS, solvent=SOLVENT)
O, min_ov = steer.mbar_overlap(windows, centers, K, T)                 # pymbar window-overlap matrix + weakest adjacent link
print(f'{NWIN} windows sampled. connected? min adjacent overlap = {min_ov:.3f}  ({"yes" if min_ov > 0.03 else "add windows"})')

### 3.5b  What umbrella sampling actually produces — *before* any statistics
Here is the raw output. **Left:** each biased window is a histogram of the CV recorded while that window's restraint held the system near its center (colored blue→red = folded→open along the ladder; ticks along the bottom mark the `centers`). Read two things off it — neighbours **overlap** (their histograms share CV range, the bridge MBAR needs) and together they **tile** the whole coordinate, out into the region the **unbiased reference** (grey fill) never reaches. That grey fill is exactly where the cross-check in 3.5c can bite, and past its edge nothing independent can. **Right:** pymbar's **overlap matrix** — entry (i, j) is the probability a sample from window i could have come from window j; a solid **bright band along the diagonal** means the chain is unbroken from closed to open. The sample cell prints the weakest adjacent overlap; comfortably above ~0 means the chain holds (near zero → add windows, spaced closer).

In [ ]:
#@title §3.5b — umbrella windows + overlap figure (code)
# SEE the windows (still no reweighting): biased per-window histograms + the unbiased reference, and the
# overlap matrix that says they connect. This is the raw material MBAR consumes.
import matplotlib.cm as _cm
fig, (axW, axO) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={'width_ratios': [1.7, 1]})
_cvref = np.concatenate([cv_polyPro(md.load(f, top=TOP)) for f in ref_trajs()])       # unbiased reference (Å)
_lo = min(_cvref.min(), min(w.min() for w in windows) * 10); _hi = max(_cvref.max(), max(w.max() for w in windows) * 10)
_edges = np.linspace(_lo, _hi, 60)
axW.hist(_cvref, bins=_edges, density=True, color='0.75', alpha=0.7, label='unbiased reference (free MD)')
_cols = _cm.turbo(np.linspace(0, 1, len(windows)))
for _w, _c, _col in zip(windows, centers, _cols):
    axW.hist(_w * 10, bins=_edges, density=True, histtype='step', color=_col, lw=1.3)
    axW.plot(_c * 10, 0, marker='^', color=_col, ms=6, clip_on=False)                  # restraint center
axW.set_xlabel('indole→polyPro distance (Å)'); axW.set_ylabel('sampled density (per window)')
axW.set_title(f'{len(windows)} biased windows tile the coordinate (blue→red = folded→open)'); axW.legend(fontsize=8, loc='upper right')
_im = axO.imshow(O, origin='lower', cmap='viridis', vmin=0)
axO.set_xlabel('window j'); axO.set_ylabel('window i'); axO.set_title(f'overlap matrix (min adj = {min_ov:.2f})')
fig.colorbar(_im, ax=axO, fraction=0.046, label='overlap  p(i→j)'); plt.tight_layout(); plt.show()

### 3.5c  Stitching the windows into one curve — MBAR
**MBAR — divide the bias back out.** MBAR (multistate Bennett acceptance ratio; **Shirts & Chodera**, *J. Chem. Phys.* **129**, 124105 (2008), doi:10.1063/1.2978177) is the reweighting that removes the bias: given every window's samples and the known restraint energy of each, it solves self-consistently for the per-window free energies and assigns each sample an **unbiased weight**; histogramming those weights gives the **PMF**. It's the statistically-optimal, binless successor to **WHAM** — the histogram-based classic, if you want to dig into the underlying machinery: **Kumar *et al.***, *J. Comput. Chem.* **13**, 1011 (1992), doi:10.1002/jcc.540130812. We compute the PMF from a **compact, self-contained MBAR** (`steer.mbar_pmf`, ~30 readable lines) so the algorithm is visible rather than a black box — validated to reproduce pymbar's window free energies to machine precision. For the **uncertainty** we *bootstrap* (resample each window's frames, recompute the PMF, take the per-bin spread) rather than lean on any solver's built-in error bar: those are *asymptotic* estimates that assume good overlap and uncorrelated samples, and turn optimistic exactly when a run is short or poorly overlapped — the failure mode this whole section is about (**Klimovich, Shirts & Mobley**, *J. Comput.-Aided Mol. Des.* **29**, 397 (2015), doi:10.1007/s10822-015-9840-9, is the practical guide).

That's a lot of machinery in one breath, but the core is just algebra. A window with harmonic restraint $w(x)=\tfrac12 k (x-c)^2$ samples the **biased** distribution

$$P_{\text{bias}}(x)\;\propto\;P(x)\,e^{-\beta\,w(x)},\qquad\beta=\tfrac{1}{k_\mathrm{B}T},$$

the true $P(x)$ tilted by the restraint's Boltzmann factor. Take $-k_\mathrm{B}T\ln(\cdot)$ of both sides and rearrange:

$$\underbrace{-k_\mathrm{B}T\ln P_{\text{bias}}(x)\;-\;w(x)}_{\text{from the window's histogram}}\;=\;\underbrace{-k_\mathrm{B}T\ln P(x)}_{G(x)\ \text{(unbiased)}}\;+\;f_k.$$

So **take a window's log-histogram and subtract its restraint energy** $w(x)$ — that *is* dividing out the Boltzmann factor — and you recover the unbiased free energy $G(x)$, **up to a window-specific constant $f_k$**. Those constants are the whole game: where two windows **overlap**, their recovered $G(x)$ must agree, which pins the offset $f_j-f_i$ between them; shift one onto the other, average where they overlap, and the two become one curve. **MBAR** solves all the $f_k$ at once, self-consistently, for the whole ladder. The cell below does it by hand for one well-chosen pair:

In [ ]:
# TOY: MBAR's core by hand, on ONE pair of windows (the algebra is in the markdown just above).
# For each window: unbias = subtract its restraint energy w(x); the overlap pins the offset f between
# the two; average where they overlap -> one curve.
kT = 0.00831446261815324 * T
# pick a well-behaved adjacent pair to demonstrate on: clearly ordered AND overlapping. This auto-avoids
# the steep barrier windows, whose biased distributions skew downhill off-centre and can even land out of order.
def _score(a, b):
    ma, mb = np.median(windows[a]), np.median(windows[b])
    if mb <= ma: return -1.0                                       # require order: b sampled farther out than a
    lo = max(np.percentile(windows[a], 5), np.percentile(windows[b], 5))
    hi = min(np.percentile(windows[a], 95), np.percentile(windows[b], 95))
    return min((mb - ma) * 10, (hi - lo) * 10)                     # reward BOTH separation and overlap (Å)
i, j = max([(k, k + 1) for k in range(NWIN - 1)], key=lambda p: _score(*p))   # clearest ordered+overlapping pair
xi, xj = windows[i] * 10, windows[j] * 10                          # each window's CV samples (Å)
_lo = min(np.percentile(xi, 5), np.percentile(xj, 5)); _hi = max(np.percentile(xi, 95), np.percentile(xj, 95))
e = np.linspace(_lo, _hi, 8); m = 0.5 * (e[:-1] + e[1:])           # a SHARED coarse grid (windows store only ~80 frames)
ci = np.histogram(xi, bins=e)[0]; cj = np.histogram(xj, bins=e)[0]  # per-bin counts (weight the combine below)
def _G(x, c, cnt):                                                # window's unbiased FE on the shared grid; NaN where unsampled
    lnP = -kT * np.log(np.histogram(x, bins=e, density=True)[0] + 1e-12)   # biased free energy -kT lnP_bias
    g = (lnP - 0.5 * K * ((m / 10) - c) ** 2).copy()              # subtract w(x) = divide out the Boltzmann factor
    g[cnt < 3] = np.nan; return g                                 # keep only bins this window actually sampled
gi = _G(xi, centers[i], ci); gj = _G(xj, centers[j], cj)
lnPj = -kT * np.log(np.histogram(xj, bins=e, density=True)[0] + 1e-12); wj = 0.5 * K * ((m / 10) - centers[j]) ** 2  # for the table
ov = np.isfinite(gi) & np.isfinite(gj)                            # bins BOTH windows fill = the overlap
off = float(np.nanmean(gj[ov] - gi[ov])) if ov.any() else 0.0     # the constant the overlap pins (window free-energy diff)
gjs = gj - off                                                    # window j, shifted onto window i
comb = np.where(np.isfinite(gi), gi, np.nan)                      # COMBINE -> one curve: window i where it has data,
comb = np.where(np.isfinite(gjs) & ~np.isfinite(gi), gjs, comb)   #   window j (shifted) where only it does,
comb[ov] = (ci[ov] * gi[ov] + cj[ov] * gjs[ov]) / (ci[ov] + cj[ov])  #   and the count-weighted mean in the overlap

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.7))
ax[0].hist(xi, bins=12, density=True, color='C0', alpha=0.5, label=f'window {i} (inner)')
ax[0].hist(xj, bins=12, density=True, color='C1', alpha=0.5, label=f'window {j} (outer)')
if ov.any(): ax[0].axvspan(m[ov].min(), m[ov].max(), color='0.85', zorder=0, label='overlap')
ax[0].set_xlim(_lo - 0.3, _hi + 0.3); ax[0].set_title('1 · two overlapping windows'); ax[0].set_xlabel('CV (Å)'); ax[0].set_ylabel('density'); ax[0].legend(fontsize=7)
ax[1].plot(m, gi, 'o-', color='C0', lw=1.6, label=f'window {i}: unbiased')
ax[1].plot(m, gj, 's--', color='C1', lw=1.4, alpha=0.6, label=f'window {j}: own zero')
ax[1].plot(m, gjs, 'D-', color='C3', lw=1.6, label=f'window {j}: shifted by f={off:.1f}')
if ov.any():                                                      # draw the offset as an arrow at the overlap centre
    _xm = float(np.mean(m[ov])); _fin = np.isfinite(gj); _y0 = float(np.interp(_xm, m[_fin], gj[_fin]))
    ax[1].annotate('', xy=(_xm, _y0 - off), xytext=(_xm, _y0), arrowprops=dict(arrowstyle='-|>', color='0.35', lw=1.6))
    ax[1].text(_xm, _y0 - off / 2, '  −f', color='0.25', fontsize=10, va='center'); ax[1].axvspan(m[ov].min(), m[ov].max(), color='0.85', zorder=0)
ax[1].set_title('2 · unbias (−w), align by the offset f'); ax[1].set_xlabel('CV (Å)'); ax[1].set_ylabel('free energy (kJ/mol)'); ax[1].legend(fontsize=7)
ax[2].plot(m, gi, 'o-', color='C0', lw=1, alpha=0.3); ax[2].plot(m, gjs, 'D-', color='C3', lw=1, alpha=0.3)   # the two aligned pieces, faded
ax[2].plot(m, comb, '-', color='k', lw=2.4, label='combined (one curve)')
if ov.any(): ax[2].axvspan(m[ov].min(), m[ov].max(), color='0.85', zorder=0)
ax[2].set_title('3 · combine → one curve (avg where they overlap)'); ax[2].set_xlabel('CV (Å)'); ax[2].set_ylabel('free energy (kJ/mol)'); ax[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

# the NUMBERS behind the stitch: the overlap bins, step by step (what the plot did)
print(f'overlap bins: subtract w(x) to unbias window {j}, then subtract f = {off:.1f} kJ/mol to meet window {i}')
print(f'  CV(Å)   −kT·lnP    w(x)    G=lnP−w    G−f    (w{i} G here)')
for _k in np.where(ov)[0]:
    print(f'  {m[_k]:5.2f}   {lnPj[_k]:7.2f}   {wj[_k]:6.2f}   {gj[_k]:7.2f}   {gj[_k] - off:6.2f}     {gi[_k]:6.2f}')
print(f'  -> G−f tracks window {i}\'s G in the overlap; averaging the two there gives the single black curve')
print('     in panel 3. MBAR does exactly this for ALL windows at once, solving every offset self-consistently.')

In [ ]:
# REWEIGHT (all NWIN windows at once): what the toy did for two, MBAR does for every window, solving all
# the offsets self-consistently -> ONE unbiased PMF. Cross-checked in the closed well against the SAME
# honest -kT lnP reference used in 3.0 (single-sourced -> identical masking + band).
kT = 0.00831446261815324 * T
xA, pmf, err = steer.mbar_pmf_bootstrap(windows, centers, K, T)        # self-contained MBAR (== pymbar f_k) + WITHIN-run bootstrap error
_cv_per = [cv_polyPro(md.load(f, top=TOP)) for f in ref_trajs()]        # per-seed reference CV (Å)
mc, G_ref, band_ref, _ = steer.reference_pmf(_cv_per, T, nbins=30)      # honest reference: flaring band, contiguous mask
# Both curves are free energies only up to a constant -> reference them to the SAME zero: the folded
# WELL BOTTOM (deepest, best-sampled point; the natural folded reference state). reference_pmf already
# zeros there, so anchor the MBAR PMF to the same point -> the well agrees by construction and any
# leftover gap is honest SHAPE disagreement, not an alignment artifact. (Matching a wide window's mean
# instead mis-set the well depth whenever MBAR's unconverged wall shape differed, floating the PMF off
# the very basin it should sit in.)
_xw = mc[int(np.nanargmin(G_ref))]                                     # reference well minimum (~4.7 Å)
_fin = np.isfinite(pmf)                                                # guard: a degenerate single run can return an all-NaN PMF
_near = np.isfinite(G_ref) & (np.abs(mc - _xw) < 0.6) & (mc >= (xA[_fin].min() if _fin.any() else xA.min()))   # snug window at the well BOTTOM
_shift = (np.interp(mc[_near], xA[_fin], pmf[_fin]).mean() - G_ref[_near].mean()) if (_near.any() and _fin.any()) else 0.0
pmf = pmf - _shift                                                     # MBAR PMF zeroed at the folded well

plt.figure(figsize=(6.8, 4.3))
plt.errorbar(xA, pmf, yerr=err, fmt='-o', ms=3, color='navy', label='MBAR PMF (this single umbrella run)')
plt.plot(mc, G_ref, '--', color='firebrick', label='−k$_B$T ln P (unbiased ref, closed well)')
plt.fill_between(mc, G_ref - band_ref, G_ref + band_ref, color='firebrick', alpha=0.15,
                 label='reference uncertainty (seed spread; flares as seeds thin)')
plt.xlabel('indole→polyPro distance (Å)'); plt.ylabel('free energy (kJ/mol)')
plt.title('Free energy vs the cage-opening coordinate — one run, cross-checked in the well'); plt.legend(fontsize=8)
plt.ylim(bottom=min(-2.0, (float(np.nanmin(pmf)) - 1) if _fin.any() else -2.0)); plt.show()
if _fin.any():
    print(f'top of the ramp (~{xA[np.nanargmax(pmf)]:.1f} Å) THIS run ≈ {np.nanmax(pmf):.0f} kJ/mol = {np.nanmax(pmf)/kT:.1f} k_BT.')
    print('That is the cost to CRANK the CV out to there -- NOT a cage-opening ΔG: no distinct open state to switch to (see 3.5d).')
else:
    print('This umbrella run did not tile the coordinate -- the windows stayed bunched near the folded well, so')
    print('there is no profile to read. Re-run the sample cell above; a single short umbrella pass on this slow,')
    print('collective coordinate is unreliable -- exactly the caution 3.5d is about.')

### 3.5d  Reading the profile — and the real lesson
**Reading the profile.** The PMF **rises** out of the folded well and climbs as the restraint forces the coordinate outward. We **cross-check** the part we can: in the closed well, where the unbiased reference actually sampled, the PMF should sit on top of the −k_BT·ln P(CV) from those trajectories (§3.0's curve) — evidence MBAR didn't invent the folded basin. But be honest about what that buys: the reference is itself only six 10 ns runs, **not a converged free energy** — agreement there means the two estimates are *consistent*, not that either is *true*. Six short runs can share the same blind spots, and the seed band only sees where they *disagree*, never an event all six of them miss. So the cross-check is a **sanity floor, not ground truth**. And past the well there is **nothing left to check against** at all: the reference only samples out to ~6 Å, and even there the far edge is a *single* seed (§3.0's band fans apart), so the open arm floats free of any independent measurement.

**Honesty — the real lesson, and what this number is *not*.** Read the profile for what it *is*: **one single umbrella run**, whose error bars only bootstrap the frames *within* that run — so they can't see the variance that actually matters. Rerun the cell and the curve moves: the folded well stays put, but the far end wanders by many kJ/mol. Pool many *independent* replicas and that scatter **collapses onto a smooth climb** — which is good news and bad news. Good, because it means the run-to-run spread is mostly **undersampling**, the kind more sampling cures. Bad, because what it collapses *to* is a smooth shoulder with **no second basin** — and that is the deepest catch: **even with the ladder capped at lid-off, this is not a cage-opening free energy.** A free-energy *difference* needs two well-defined states; we have exactly one clean basin (folded) and, past it, a **ramp** with no distinct open state to land in. So the top of the curve is not 'the cost to open the cage' — it is the cost to **crank the coordinate out to ~8 Å**, a cutoff we chose. We never watched the system *switch states*, because along this coordinate there is no second state to switch to. The number is real; the label 'ΔG of opening' is not earned.

Two transferable lessons outlive Trp-cage. **(1)** A converged-*looking* free energy can still be wrong — you catch it with **rerun spread and validation against unbiased data**, not the solver's built-in checks (in the closed well, where six reference seeds agree, we can corroborate; past it, nothing independent reaches). **(2)** Before you call a PMF value a ΔG, **make sure your endpoint is actually a state** — a basin, not a shoulder — or you have measured the cost to reach a coordinate *value*, not a transition between states.

**The analysis is tutorial-grade too, not just the sampling.** MBAR itself is the *right* tool, correctly implemented — its per-window free energies match pymbar's to ~1e-8, and the open arm is genuinely *sampled* by the windows (biased, but real data — not extrapolation). What's cut short is the statistical hygiene around it: we feed all ~80 frames per window as if independent (they're correlated, so the *effective* count is a handful), we don't detect and discard each window's un-equilibrated start, and the bootstrap resamples those correlated frames as independent — so even the within-run error bar flatters itself. Deeper still, MBAR assumes each window is sampled *at equilibrium*; a single, 12-ps-equilibrated window with a slow orthogonal mode is not, which biases the f_k themselves. None of this is *misusing* MBAR — it is the standard umbrella→PMF workflow, run at a scale a real study would beat by 100–1000× in sampling, with proper decorrelation, equilibration detection, and independent replicas. The tool is right and exact; the scale and the hygiene are what keep this a demonstration.

## 3.6  A transition you *can* check — the Asp9–Arg16 salt bridge
Everything in §3.5 came with an asterisk: past the folded well we had **no way to check the PMF**, because unbiased MD never went there. The surface **Asp9–Arg16 salt bridge** (the folded latch from §3.3) is the opposite case — and that is exactly what makes it worth one more run. In the plain 10 ns reference this contact **opens and re-closes on its own**: its distance wanders from ~3.5 Å out past 8–9 Å and back, so a real fraction of unbiased frames sit with the bridge broken. This coordinate has genuine two-state character (contact ↔ separated) **and** unbiased data spanning the whole range.

That cuts two ways, and the tension is the whole point:
- **You don't actually *need* enhanced sampling to reach this motion.** Plain MD already crosses it in 10 ns — §3.1's wall was specific to the *cage*, not to every coordinate. Umbrella still helps (six short seeds *reach* the transition but don't fully *converge* its populations, and tiling it with windows is a more efficient, better-controlled estimate) — but the transition is *accessible* either way.
- **Because plain MD reaches it, we can validate the method here.** We run the *exact same machinery* as §3.5 — a ladder of umbrella windows → MBAR → a PMF — then lay it directly on top of the unbiased −k_BT·ln P. Where the cage handed us a number we could never check, here there is **ground truth across the whole coordinate.**

Same steps as §3.5, one difference: this time we get to see whether the answer is right. (Same cost, too — this cell is real MD and takes a few minutes, like §3.5's.)

In [ ]:
#@title §3.6 — salt-bridge umbrella sample (same machinery as §3.5; code)
# SAMPLE: the §3.5 umbrella machinery, now on the salt-bridge distance -- the coordinate plain MD already
# explores. Ladder from the folded contact out through the separated state; each window settles then samples.
_asp = mdtop.select('resSeq 9 and name OD1 OD2'); _arg = mdtop.select('resSeq 16 and name NH1 NH2 NE')
cv_sb = lambda tr: np.linalg.norm(tr.xyz[:, _asp].mean(1) - tr.xyz[:, _arg].mean(1), axis=1) * 10   # Å (matches the biased centroid CV)
fac_sb, delta_sb = CVS['salt bridge']; K = 4000.0; T = TEMP
folded_sb = float(np.linalg.norm(x0.xyz[0, _asp].mean(0) - x0.xyz[0, _arg].mean(0)))                 # nm
NWIN_SB = 15                                                        # ~6 Å span (wider than the cage) -> a few more windows for overlap
centers_sb = np.linspace(folded_sb, folded_sb + delta_sb, NWIN_SB)
windows_sb = steer.run_umbrella(pdb, fac_sb, centers_sb, T, PLATFORM, k=K, equil_ps=EQUIL_PS, sample_ps=WIN_PS, solvent=SOLVENT, independent=True)
O_sb, min_ov_sb = steer.mbar_overlap(windows_sb, centers_sb, K, T)
print(f'{NWIN_SB} salt-bridge windows sampled ({centers_sb[0]*10:.1f}->{centers_sb[-1]*10:.1f} Å). '
      f'min adjacent overlap = {min_ov_sb:.3f}  ({"connected" if min_ov_sb > 0.03 else "add windows"})')

In [ ]:
#@title §3.6 — MBAR PMF vs the unbiased reference (code)
# THE CHECK: run the SAME MBAR as §3.5, then lay it on the UNBIASED free energy -- the comparison the cage
# never allowed. Left: biased windows tile the coordinate, over the reference (grey) that already spans it.
# Right: MBAR PMF vs -kT lnP from plain MD, both anchored at the folded well -> agreement is shape, not offset.
import matplotlib.cm as _cm
kT = 0.00831446261815324 * T
_sb_per = [cv_sb(md.load(f, top=TOP)) for f in ref_trajs()]                  # per-seed unbiased salt-bridge CV (Å)
_sb_all = np.concatenate(_sb_per); _frac_open = (_sb_all > 6).mean() * 100    # how often plain MD breaks the bridge
import json as _json                                                          # timescale aside (ties back to §2.6):
_dtref = float(_json.load(open(ensure_file(os.path.join(REF, "run_meta.json")))).get("reference_stride_ps", 1))  # ps/frame
_taus = [steer.integrated_autocorr_time(v) for v in _sb_per]                  # per-seed tau (frames); per-seed avoids cross-seam jumps
_tau_ps = float(np.mean(_taus)) * _dtref                                      # mean decorrelation time of the salt-bridge distance (ps)
_neff_sb = float(sum(len(v) / (2 * t) for v, t in zip(_sb_per, _taus)))       # INDEPENDENT looks across the 6x10 ns reference (§2.6 estimator)
mc_sb, G_sb, band_sb, _ = steer.reference_pmf(_sb_per, T, nbins=30)           # honest reference FE (same helper as 3.0 / 3.5c)
xA_sb, pmf_sb, err_sb = steer.mbar_pmf_bootstrap(windows_sb, centers_sb, K, T)
_xw = mc_sb[int(np.nanargmin(G_sb))]                                         # folded well minimum
_near = np.isfinite(G_sb) & (np.abs(mc_sb - _xw) < 0.6) & (mc_sb >= xA_sb.min())
_shift = (np.interp(mc_sb[_near], xA_sb, pmf_sb).mean() - G_sb[_near].mean()) if _near.any() else 0.0
pmf_sb = pmf_sb - _shift                                                     # zero the MBAR PMF at the same well as the reference
fig, (axW, axP) = plt.subplots(1, 2, figsize=(12, 4.3))
_lo = min(_sb_all.min(), min(w.min() for w in windows_sb) * 10)
_hi = min(12.0, max(_sb_all.max(), max(w.max() for w in windows_sb) * 10))   # cap the sparse far tail: past ~12 Å the reference barely registers and just squishes the histograms
_edges = np.linspace(_lo, _hi, 60)
axW.hist(_sb_all, bins=_edges, density=True, color='0.75', alpha=0.7, label='unbiased reference (plain MD)')
for _w, _c, _col in zip(windows_sb, centers_sb, _cm.turbo(np.linspace(0, 1, NWIN_SB))):
    axW.hist(_w * 10, bins=_edges, density=True, histtype='step', color=_col, lw=1.2)
    axW.plot(_c * 10, 0, marker='^', color=_col, ms=6, clip_on=False)
axW.set_xlabel('Asp9–Arg16 distance (Å)'); axW.set_ylabel('sampled density')
axW.set_title(f'windows tile it — but so does plain MD ({_frac_open:.0f}% of frames > 6 Å)'); axW.legend(fontsize=8)
axW.set_xlim(3, 12)   # crop the sparse >12 Å tail so the folded-contact region isn't squished
axP.errorbar(xA_sb, pmf_sb, yerr=err_sb, fmt='-o', ms=3, color='navy', label='MBAR PMF (one umbrella run)')
axP.plot(mc_sb, G_sb, '--', color='firebrick', label='−k$_B$T ln P (unbiased reference)')
axP.fill_between(mc_sb, G_sb - band_sb, G_sb + band_sb, color='firebrick', alpha=0.15, label='reference uncertainty')
axP.set_xlim(3, 10); axP.set_xlabel('Asp9–Arg16 distance (Å)'); axP.set_ylabel('free energy (kJ/mol)')
axP.set_title('umbrella vs ground truth'); axP.legend(fontsize=8)
plt.tight_layout(); plt.show()
_win = np.isfinite(G_sb) & (mc_sb >= 6) & (mc_sb <= 8.5)
_z = lambda v: 0.0 if abs(v) < 0.05 else v                                   # clamp tiny magnitudes so a small negative never prints as "-0"
if _win.any():
    _xe = mc_sb[_win][int(np.nanargmax(G_sb[_win]))]; _er = float(np.nanmax(G_sb[_win])); _eu = float(np.interp(_xe, xA_sb, pmf_sb))
    print(f'breaking the contact (folded -> ~{_xe:.1f} Å): reference {_z(_er):+.1f} vs umbrella {_z(_eu):+.1f} kJ/mol '
          f'(this run climbs to a peak of {np.nanmax(pmf_sb):.1f} kJ/mol overall -- a flat ~0 here would just be one noisy replica).')
print(f'timescale (the §2.6 estimator): the salt-bridge distance decorrelates every ~{_tau_ps:.0f} ps, so the 6x10 ns '
      f'reference holds ~{_neff_sb:.0f} independent looks at it -- enough to SEE the transition (why we can check it), '
      f'not so many that umbrella efficiency is moot (why it still helps).')
print('past ~9 Å both curves climb together: Asp9 and Arg16 are 7 residues apart, so that is the backbone tether, '
      'not a second barrier -- the same "the CV runs past the event" trap as the cage.')

### 3.6b  What the check tells us — and the shape of the whole section
The two curves **agree where it counts.** Breaking the contact costs only a few kJ/mol (a couple k_BT — a weak surface latch, consistent with §3.3), and the MBAR PMF tracks the reference **inside its seed band** across the basin-exit region. Same machinery, same cheap scale as the cage — but here we can *see* it landed on the truth. The far (>9 Å) climb is **not** a barrier: Asp9 and Arg16 are seven residues apart, so pulling them further eventually fights the backbone tether; both curves rise there together, the coordinate simply running past the physical event (the same *capture ≠ saturate* trap as the cage's distance CV).

**Why was it reachable at all?** The printout answers with the §2.6 tool: the salt-bridge distance decorrelates on a timescale short enough that the 6 × 10 ns reference holds a healthy N_eff of *independent* looks at it — computed with the same integrated-autocorrelation-time estimator that placed §2.6's 2τ line. That is what *plain MD samples it* means numerically, and it is the exact mirror of §2.6's verdict on the cage, whose opening mode was too slow to fit inside 10 ns even once. Enough independent looks to rough out the free energy is precisely what makes this the checkable case; too few, and we'd be right back in the cage's blind spot.

That agreement is the payoff, and it comes with a caveat worth stating plainly. Put the two sections side by side:

| | unbiased MD reaches it? | enhanced sampling… | can you validate it? |
|---|---|---|---|
| **Cage opening (§3.5)** | no (never opens in 10 ns) | **necessary** | **no** — no ground truth past the well |
| **Salt bridge (§3.6)** | yes (opens & re-closes) | *optional* (buys efficiency) | **yes** — and it checks out |

The rows are mirror images, and the uncomfortable generalization falls out of the table: **enhanced sampling is most indispensable exactly where it is hardest to validate.** The salt bridge does *not* prove the cage PMF is correct — nothing can, that's the point. What it proves is that the **method** — umbrella windows stitched by MBAR — recovers the right free energy on a case we *can* see. That is the most reassurance you can honestly buy: validate the machinery where ground truth exists, then apply it, eyes open, where it doesn't.

And keep the framing from §3.5 in view: this salt-bridge run is **still a cheap demo**. It agrees with the reference not because we sampled it heroically but because the coordinate is easy *and* checkable. For the cases that actually need enhanced sampling you don't get that luxury — there you lean on convergence diagnostics, multiple coordinates, and replica methods like replica exchange, never a single short run.